New Notebook Created by Jupyter MCP Server

In [1]:
import re, sys
sys.path.insert(0, '/home/tiencd/rf-worldpose')

with open('/home/tiencd/rf-worldpose/scripts/assign_subject_splits.py') as f:
    src = f.read()

lines = src.splitlines()
print(f'Script: {len(lines)} lines')

functions = re.findall(r'^def (\w+)', src, re.MULTILINE)
print('Functions:', functions)

protocols = re.findall(r'Protocol \d+|WiPose', src)
print('Protocols:', sorted(set(protocols)))

Script: 163 lines
Functions: ['_mmfi_proto1_split', '_mmfi_split', '_wipose_split', 'assign_mmfi', 'assign_wipose', 'process', 'main']
Protocols: ['Protocol 1', 'Protocol 2', 'WiPose']


In [2]:
# Simulate MM-Fi Protocol 2: cross-subject split
# 4 envs x 40 subjects x 27 actions
import hashlib
from collections import Counter

def subject_last_digit(s):
    return int(str(s)[-1])

envs = [1, 2, 3, 4]
subjects = list(range(1, 41))
actions = list(range(1, 28))

split_counts = Counter()
subject_split_map = {}

for e in envs:
    for s in subjects:
        d = subject_last_digit(s)
        if d in (5, 0):
            split = 'test'
        elif d == 8:
            split = 'val'
        else:
            split = 'train'
        subject_split_map[(e, s)] = split

for e in envs:
    for s in subjects:
        for a in actions:
            split_counts[subject_split_map[(e, s)]] += 1

total = sum(split_counts.values())
print('Protocol 2 - Cross-subject split simulation')
print('-' * 45)
for sp in ['train', 'val', 'test']:
    n = split_counts[sp]
    print(f'  {sp:6s}: {n:5d} samples  ({100*n/total:.1f}%)')
print(f'  Total : {total:5d} samples')

Protocol 2 - Cross-subject split simulation
---------------------------------------------
  train :  3024 samples  (70.0%)
  val   :   432 samples  (10.0%)
  test  :   864 samples  (20.0%)
  Total :  4320 samples


In [3]:
# Protocol 1: within-subject hash-based split
import hashlib
from collections import Counter

def proto1_split(sample_id, seed=42):
    h = hashlib.md5(f'{seed}:{sample_id}'.encode()).hexdigest()
    v = int(h[:8], 16) / 0xFFFFFFFF
    if v < 0.80:
        return 'train'
    elif v < 0.90:
        return 'val'
    else:
        return 'test'

counts = Counter()
for e in range(1, 5):
    for s in range(1, 41):
        for a in range(1, 28):
            sid = f'E{e}_S{s}_A{a}'
            counts[proto1_split(sid)] += 1

total = sum(counts.values())
print('Protocol 1 - Within-subject hash split simulation')
print('-' * 50)
for sp in ['train', 'val', 'test']:
    n = counts[sp]
    print(f'  {sp:6s}: {n:5d} samples  ({100*n/total:.1f}%)')
print(f'  Total : {total:5d} samples')
print()
print('NOTE: Same subject can appear in multiple splits (subject leakage)')

Protocol 1 - Within-subject hash split simulation
--------------------------------------------------
  train :  3425 samples  (79.3%)
  val   :   445 samples  (10.3%)
  test  :   450 samples  (10.4%)
  Total :  4320 samples

NOTE: Same subject can appear in multiple splits (subject leakage)


In [4]:
# Compare protocols side by side
proto2 = {'train': 3024, 'val': 432, 'test': 864, 'total': 4320}
proto1 = {'train': 3425, 'val': 445, 'test': 450, 'total': 4320}

print(f'{'Split':<10} {'Protocol 1':>14} {'Protocol 2':>14}')
print('-' * 40)
for k in ['train', 'val', 'test']:
    p1 = proto1[k]
    p2 = proto2[k]
    p1_pct = 100*p1/4320
    p2_pct = 100*p2/4320
    print(f'{k:<10} {p1:5d} ({p1_pct:.0f}%)   {p2:5d} ({p2_pct:.0f}%)')
print('-' * 40)
print(f'{'Total':<10} {4320:5d}          {4320:5d}')
print()
print('Protocol 1: honest test = 20%, subject never leaks across splits')
print('Protocol 2: honest test = 10%, some subject leakage possible')
print()
print('=> Protocol 2 is harder, closer to real-world deployment')

Split          Protocol 1     Protocol 2
----------------------------------------
train       3425 (79%)    3024 (70%)
val          445 (10%)     432 (10%)
test         450 (10%)     864 (20%)
----------------------------------------
Total       4320           4320

Protocol 1: honest test = 20%, subject never leaks across splits
Protocol 2: honest test = 10%, some subject leakage possible

=> Protocol 2 is harder, closer to real-world deployment


In [1]:
import sys, os
sys.path.insert(0, '/home/tiencd/rf-worldpose')

# Đọc nội dung script
with open('/home/tiencd/rf-worldpose/scripts/assign_subject_splits.py') as f:
    content = f.read()

lines = content.splitlines()
print(f'📄 assign_subject_splits.py: {len(lines)} dòng')
print('\n--- 30 dòng đầu ---')
print('\n'.join(lines[:30]))

📄 assign_subject_splits.py: 163 dòng

--- 30 dòng đầu ---
#!/usr/bin/env python3
"""Assign train/val/test splits to Gold metadata.

Supports two protocols for MM-Fi:

  Protocol 2 — Cross-subject (default, honest evaluation)
    MM-Fi (sample_id = ``E{e}_S{s}_A{a}``)
    Balanced across all 4 environments:
      test : 2 subjects / env  (subject last digit in {5, 0})  → 8 subjects (~20%)
      val  : 1 subject  / env  (subject last digit == 8)        → 4 subjects (~10%)
      train: remaining 28 subjects                              (~70%)
    Subject never appears in two splits.

  Protocol 1 — Within-subject (sequence-level, matches most papers)
    Split by full sequence id ``E{e}_S{s}_A{a}`` hash → 80/10/10.
    Same subject's different actions can appear in different splits (subject leakage),
    but all windows from the same sequence stay together (no window leakage).
    Gives ~2-3× better MPJPE than cross-subject.

  WiPose (NjtechCVLab)
    Keep the dataset's native Train/Test

In [2]:
import re
from collections import Counter

# Phân tích cấu trúc của script
with open('/home/tiencd/rf-worldpose/scripts/assign_subject_splits.py') as f:
    src = f.read()

# Tìm tất cả functions
functions = re.findall(r'^def (\w+)', src, re.MULTILINE)
print('\ud83d\udd27 Các hàm trong script:')
for fn in functions:
    print(f'  - {fn}()')

# Tìm các protocol
protocols = re.findall(r'Protocol \d+|WiPose|PROTOCOL', src)
print(f'\n\ud83d\udcca Protocols được hỗ trợ: {set(protocols)}')

# Tìm các split ratio
ratios = re.findall(r'(?:train|val|test)[^\n]*?\d+[%/]\d*', src, re.IGNORECASE)
print('\n\ud83c\udfaf Split ratios tìm được:')
for r in ratios[:5]:
    print(f'  {r.strip()}')

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/tiencd/.local/lib/python3.12/site-packages/jupyter_client/session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/tiencd/.local/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 550, in _run_callback
    f = callback(*args, **kwargs)
        ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tiencd/.local/lib/python3.12/site-packages/ipykernel/iostream.py", line 245, in _handle_event
    event_f()
  File "/home/tiencd/.local/lib/python3.12/site-packages/ipykernel/iostream.py", line 730, in _flush
    self.session.send(
  File "/home/tiencd/.local/lib/python3.12/site-packages/jupyter_c

^^^
  File "/home/tiencd/.local/lib/python3.12/site-packages/jupyter_client/session.py", line 727, in serialize
    content = self.pack(content)
              ^^^^^^^^^^^^^^^^^^
  File "/home/tiencd/.local/lib/python3.12/site-packages/jupyter_client/session.py", line 111, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surrogates not allowed
ERROR:tornado.application:Exception in callback functools.partial(<function ZMQStream._update_handler.<locals>.<lambda> at 0x738a83637b00>)
Traceback (most recent call last):
  File "/home/tiencd/.local/lib/python3.12/site-packages/jupyter_client/session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surrogates not allowed

During handling of the above exception